In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyarrow.dataset as ds

%matplotlib inline


# ============================================================
# Read alert times only
# ============================================================

alerts_ds = ds.dataset(
    "antares_data/alerts",
    format="parquet",
)

table = alerts_ds.to_table(
    columns=["mjd"]
)

mjd = table["mjd"].to_numpy()

# Remove missing / invalid values
mjd = mjd[np.isfinite(mjd)]


# ============================================================
# Convert MJD to calendar date
#
# MJD = 40587 corresponds to 1970-01-01
# ============================================================

dates = pd.to_datetime(
    mjd - 40587.0,
    unit="D",
    origin="unix",
    utc=True,
)


# ============================================================
# Basic information
# ============================================================

print(f"Number of LSST alerts: {len(mjd):,}")
#print(f"Earliest alert: {dates.min()}")
#print(f"Latest alert:   {dates.max()}")

print(
    "Earliest alert:",
    dates.min().strftime("%Y-%m-%d %H:%M:%S UTC")
)

print(
    "Latest alert:  ",
    dates.max().strftime("%Y-%m-%d %H:%M:%S UTC")
)


# ============================================================
# Histogram
# ============================================================

fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(
    dates,
    bins=50,
)

ax.set_xlabel("Alert date")
ax.set_ylabel("Number of alerts")
ax.set_title("Distribution of LSST alert times")

ax.tick_params(
    axis="x",
    rotation=45,
)

# Text to show on the figure
text = (
    f"Number of alerts: {len(mjd):,}\n"
    f"Earliest: {dates.min().strftime('%Y-%m-%d %H:%M:%S UTC')}\n"
    f"Latest:   {dates.max().strftime('%Y-%m-%d %H:%M:%S UTC')}"
)

ax.text(
    0.03,
    0.95,
    text,
    transform=ax.transAxes,
    va="top",
    ha="left",
)

fig.tight_layout()

#plt.show()

plt.savefig("Distribution_of_LSST_alert_times.png")

In [ ]:
# Convert to calendar day
days = pd.Series(dates).dt.floor("D")

daily_counts = (
    days.value_counts()
    .sort_index()
)


fig, ax = plt.subplots(figsize=(11, 5))

ax.bar(
    daily_counts.index,
    daily_counts.values,
    width=0.8,
)

ax.set_xlabel("Date")
ax.set_ylabel("Number of alerts per day")
ax.set_title("LSST alerts per day")

ax.tick_params(
    axis="x",
    rotation=45,
)

fig.tight_layout()

#plt.show()

plt.savefig("LSST_alerts_per_day.png")

In [ ]:
print(
    daily_counts
    .sort_values(ascending=False)
    .head(20)
)

In [ ]:
months = (
    pd.Series(dates)
    .dt.tz_localize(None)
    .dt.to_period("M")
)

monthly_counts = (
    months.value_counts()
    .sort_index()
)

print(monthly_counts)

fig, ax = plt.subplots(figsize=(10, 5))

bars = ax.bar(
    monthly_counts.index.astype(str),
    monthly_counts.values,
)

# Add count above each bar
for bar, count in zip(bars, monthly_counts.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{count:,}",
        ha="center",
        va="bottom",
    )

ax.set_xlabel("Month")
ax.set_ylabel("Number of alerts")
ax.set_title("LSST alerts per month")

ax.tick_params(
    axis="x",
    rotation=45,
)

fig.tight_layout()
#plt.show()

plt.savefig("LSST_alerts_per_month.png")